# Gold Layer: `dim_account` -- SCD-2 Dimension

| Property | Value |
|:---------|:------|
| **Target Table** | `gold.dim_account` |
| **Expected Rows** | 56,392 |
| **SCD Type** | SCD-2 (versioned history) |
| **Tracked Columns** | `AccountDesc`, `BrokerID`, `Status` |
| **MERGE Key** | `(accountid, effectivedate)` |
| **Surrogate Key** | `SK_AccountID = (AccountID x 10000) + version_number` |

### Sources
| Source | Layer | Description |
|:-------|:------|:------------|
| `bronze.customermgmt` | Bronze | XML account actions (B1: NEW, ADDACCT, UPDACCT, CLOSEACCT) |
| `bronze.account` | Bronze | CDC updates (B2/B3: I=Insert, U=Update, D=excluded) |
| `silver.batchdate` | Silver | Batch-to-date mapping for CDC timestamp resolution |

### Dimensional Joins
| Dimension | Join Type | Description |
|:----------|:----------|:------------|
| `gold.dim_customer` | **Cascading SCD-2 overlap** | Splits account versions at customer version boundaries |
| `gold.dim_broker` | **Direct join (SCD-1)** | BrokerID -> SK_BrokerID |

### Processing Steps
1. Read ALL account change events from bronze (full history)
2. Forward-fill NULLs per account (UPDACCT sends partial updates)
3. Compute initial SCD-2 dates (EffectiveDate / EndDate)
4. **Cascading SCD-2** -- overlap join with `dim_customer` to split versions
5. Assemble final DataFrame with dedup + version numbering
6. Write to Gold (MERGE / CREATE)
7. Data Quality Validation
8. Operations Logging

In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import (
    current_timestamp, col, row_number, lit, when, coalesce,
    to_timestamp, to_date, date_format, concat, concat_ws,
    count, sum as _sum, md5, lead, lag, first, last,
    expr, max as _max, min as _min, greatest, least
)
from pyspark.sql.window import Window
from pyspark.sql.types import (
    IntegerType, LongType, DecimalType, TimestampType,
    StringType, DateType, BooleanType, ByteType
)
from delta.tables import DeltaTable
from datetime import datetime

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  CONFIGURATION
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CATALOG       = "charles_schwab_retailbrokerage_dev_team_lemma"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA   = "gold"

TARGET_TABLE  = f"{CATALOG}.{GOLD_SCHEMA}.dim_account"
EXPECTED_ROWS = 56392

spark.sql(f"USE CATALOG {CATALOG}")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  PIPELINE SUMMARY
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("\n" + "="*70)
print("  GOLD.DIM_ACCOUNT -- Pipeline Configuration")
print("="*70)
print(f"  Catalog       : {CATALOG}")
print(f"  Target Table  : {TARGET_TABLE}")
print(f"  Expected Rows : {EXPECTED_ROWS:,}")
print(f"  SCD-2 Tracked : AccountDesc, BrokerID, Status")
print(f"  MERGE Key     : (accountid, effectivedate)")
print(f"  SK Formula    : (AccountID x 10000) + version_number")
print("="*70)

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 1: Read ALL Account Action History from Bronze
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  We read FULL history (not silver's latest-wins dedup) to build SCD-2.
#  Source 1: bronze.customermgmt -> XML actions (B1)
#  Source 2: bronze.account       -> CDC updates (B2/B3)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "="*70)
print("  STEP 1: Read Account Action History from Bronze")
print("="*70)

# ──────────────────────────────────────────────────────────────────────
# SOURCE 1: CustomerMgmt.xml Account Actions (Batch 1)
# ──────────────────────────────────────────────────────────────────────
print("\n  [Source 1] bronze.customermgmt (Batch 1 XML)")
print("  ActionTypes: NEW, ADDACCT, UPDACCT, CLOSEACCT")

xml_raw = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customermgmt")

xml_accounts = (
    xml_raw
    .filter(
        (col("ActionType").isin("NEW", "ADDACCT", "UPDACCT", "CLOSEACCT")) &
        (col("CA_ID").isNotNull()) & (col("CA_ID") != "")
    )
    .select(
        col("CA_ID").cast(LongType()).alias("CA_ID"),
        col("C_ID").cast(LongType()).alias("CA_C_ID"),
        col("CA_B_ID").cast(LongType()).alias("CA_B_ID"),
        col("CA_NAME").alias("CA_NAME"),
        col("CA_TAX_ST").cast(IntegerType()).alias("CA_TAX_ST"),
        when(col("ActionType") == "CLOSEACCT", lit("INAC"))
            .otherwise(lit("ACTV")).alias("CA_ST_ID"),
        to_date(col("ActionTS")).alias("action_date"),
        col("ActionType"),
        col("_batch_id").alias("_batch")
    )
)

xml_count = xml_accounts.count()
print(f"  Total rows: {xml_count:,}")
print("  ActionType distribution:")
xml_accounts.groupBy("ActionType").count().orderBy("ActionType").show(truncate=False)

print("  Sample XML account actions (first 5):")
display(xml_accounts.limit(5))

# ──────────────────────────────────────────────────────────────────────
# SOURCE 2: Account.txt CDC (Batch 2 & 3)
# ──────────────────────────────────────────────────────────────────────
print("\n  [Source 2] bronze.account (Batch 2/3 CDC)")
print("  CDC_FLAG: I=Insert, U=Update, D=excluded")

batchdate_df = (
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.batchdate")
    .select(
        col("_batch_id"),
        col("batchdate").cast(DateType()).alias("batch_date")
    )
)
print("  Batch -> Date mapping:")
batchdate_df.show(truncate=False)

bronze_account = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.account")

cdc_accounts = (
    bronze_account
    .filter(col("CDC_FLAG") != "D")
    .join(batchdate_df, on="_batch_id", how="left")
    .select(
        col("CA_ID").cast(LongType()).alias("CA_ID"),
        col("CA_C_ID").cast(LongType()).alias("CA_C_ID"),
        col("CA_B_ID").cast(LongType()).alias("CA_B_ID"),
        col("CA_NAME").alias("CA_NAME"),
        col("CA_TAX_ST").cast(IntegerType()).alias("CA_TAX_ST"),
        col("CA_ST_ID").alias("CA_ST_ID"),
        col("batch_date").alias("action_date"),
        lit("CDC").alias("ActionType"),
        col("_batch_id").alias("_batch")
    )
)

cdc_count = cdc_accounts.count()
print(f"  CDC rows after D-exclusion: {cdc_count:,}")

print("\n  Sample CDC account records (first 5):")
display(cdc_accounts.limit(5))

# ──────────────────────────────────────────────────────────────────────
# UNION BOTH SOURCES
# ──────────────────────────────────────────────────────────────────────
all_actions = xml_accounts.unionByName(cdc_accounts)
total_actions = all_actions.count()

print(f"\n  UNION complete: {total_actions:,} total actions")
print(f"    XML: {xml_count:,}  |  CDC: {cdc_count:,}")

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 2: Forward-Fill NULLs + Compute Record Hash
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  UPDACCT only sends changed fields (rest = NULL).
#  Forward-fill using LAST_VALUE(IGNORE NULLS) per CA_ID.
#
#  Every action creates a version (no hash-based dedup).
#  record_hash is computed for the Gold audit column only.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "="*70)
print("  STEP 2: Forward-Fill NULLs + Record Hash")
print("="*70)

# Window: per account, chronological order
fill_window_unbounded = (
    Window.partitionBy("CA_ID")
    .orderBy("action_date", "_batch")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# Forward-fill all business columns
filled_actions = all_actions.select(
    col("CA_ID"),
    last(col("CA_C_ID"), ignorenulls=True).over(fill_window_unbounded).alias("CA_C_ID"),
    last(col("CA_B_ID"), ignorenulls=True).over(fill_window_unbounded).alias("CA_B_ID"),
    last(col("CA_NAME"), ignorenulls=True).over(fill_window_unbounded).alias("CA_NAME"),
    last(col("CA_TAX_ST"), ignorenulls=True).over(fill_window_unbounded).alias("CA_TAX_ST"),
    last(col("CA_ST_ID"), ignorenulls=True).over(fill_window_unbounded).alias("CA_ST_ID"),
    col("action_date"),
    col("ActionType"),
    col("_batch")
)

# Compute record_hash on all mutable columns (audit trail)
changed_records = filled_actions.withColumn(
    "record_hash",
    md5(concat_ws("|",
        coalesce(col("CA_NAME"), lit("__NULL__")),
        coalesce(col("CA_B_ID").cast(StringType()), lit("__NULL__")),
        coalesce(col("CA_ST_ID"), lit("__NULL__")),
        coalesce(col("CA_TAX_ST").cast(StringType()), lit("__NULL__"))
    ))
)

versions_count = changed_records.count()

print(f"\n  Forward-fill complete:")
print(f"    Input actions : {total_actions:,}")
print(f"    Output rows   : {versions_count:,} (every action = 1 version)")
print(f"    Hash columns  : CA_NAME | CA_B_ID | CA_ST_ID | CA_TAX_ST")

print("\n  Sample after forward-fill (account with multiple actions):")
display(
    changed_records
    .filter(col("CA_ID") == 1)
    .orderBy("action_date")
)

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 3: Compute Initial SCD-2 Dates
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  EffectiveDate = action_date
#  EndDate       = LEAD(action_date) or 9999-12-31
#
#  These are BASE dates -- Step 4 will further split them
#  at customer SCD-2 boundaries (cascading).
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "="*70)
print("  STEP 3: Compute Initial SCD-2 Dates")
print("="*70)

scd_window = Window.partitionBy("CA_ID").orderBy("action_date", "_batch")

account_base_versions = (
    changed_records
    .withColumn("acct_eff_date", col("action_date"))
    .withColumn(
        "acct_end_date",
        coalesce(
            lead("action_date", 1).over(scd_window),
            lit("9999-12-31").cast(DateType())
        )
    )
    .select(
        "CA_ID", "CA_C_ID", "CA_B_ID", "CA_NAME", "CA_TAX_ST",
        "CA_ST_ID", "acct_eff_date", "acct_end_date",
        "record_hash", "_batch"
    )
)

base_count = account_base_versions.count()
unique_accounts = account_base_versions.select("CA_ID").distinct().count()

print(f"\n  Base account versions : {base_count:,}")
print(f"  Unique AccountIDs    : {unique_accounts:,}")
print(f"  Avg versions/account : {base_count / unique_accounts:.2f}")
print(f"\n  NOTE: These will be FURTHER SPLIT by customer cascading in Step 4")

print("\n  Sample base versions (Account 4 -- multi-version):")
display(
    account_base_versions
    .filter(col("CA_ID") == 4)
    .orderBy("acct_eff_date")
)

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 4: Cascading SCD-2 -- Overlap Join
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  KEY INSIGHT: Expected 56,392 > source actions 43,690
#  -> Account versions SPLIT at customer SCD-2 boundaries.
#
#  When dim_customer gets a new SK, all active accounts for
#  that customer also get new versions (cascading SCD-2).
#
#  Algorithm:
#    1. OVERLAP JOIN: account x customer where periods overlap
#    2. Compute INTERSECTION: MAX(start,start), MIN(end,end)
#    3. Then direct join to dim_broker (SCD-1, no cascade)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "="*70)
print("  STEP 4: Cascading SCD-2 -- Overlap Join")
print("="*70)

# --- dim_customer (SCD-2 -> cascading) --------------------------------
dim_customer = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_customer")
cust_count = dim_customer.count()
print(f"\n  [dim_customer] {cust_count:,} rows (SCD-2)")
print(f"  Join type: OVERLAP (cascading -- splits account versions)")
print(f"  Condition: CA_C_ID = customerid AND periods overlap")

# OVERLAP JOIN: find ALL customer versions that overlap each account version
account_cascaded = (
    account_base_versions.alias("a")
    .join(
        dim_customer.select(
            col("customerid"),
            col("sk_customerid").alias("SK_CustomerID"),
            col("effectivedate").alias("cust_eff_date"),
            col("enddate").alias("cust_end_date")
        ).alias("c"),
        on=(
            (col("a.CA_C_ID") == col("c.customerid")) &
            (col("a.acct_eff_date") < col("c.cust_end_date")) &
            (col("a.acct_end_date") > col("c.cust_eff_date"))
        ),
        how="left"
    )
)

# Compute INTERSECTION dates
account_with_customer = (
    account_cascaded
    .withColumn(
        "effectivedate",
        when(col("SK_CustomerID").isNotNull(),
             greatest(col("acct_eff_date"), col("cust_eff_date")))
        .otherwise(col("acct_eff_date"))
    )
    .withColumn(
        "enddate",
        when(col("SK_CustomerID").isNotNull(),
             least(col("acct_end_date"), col("cust_end_date")))
        .otherwise(col("acct_end_date"))
    )
    .drop("customerid", "cust_eff_date", "cust_end_date",
          "acct_eff_date", "acct_end_date")
)

cascaded_count = account_with_customer.count()
cust_null_count = account_with_customer.filter(col("SK_CustomerID").isNull()).count()
print(f"\n  Result: {cascaded_count:,} rows")
print(f"  Added from cascading: +{cascaded_count - base_count:,}")
print(f"  SK_CustomerID resolved: {cascaded_count - cust_null_count:,}")
print(f"  SK_CustomerID NULL: {cust_null_count:,} (INAC accounts)")

# --- dim_broker (SCD-1 -> direct join) --------------------------------
dim_broker = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_broker")
broker_count = dim_broker.count()
print(f"\n  [dim_broker] {broker_count:,} rows (SCD-1)")
print(f"  Join type: DIRECT (BrokerID = SK_BrokerID)")

account_with_dims = (
    account_with_customer.alias("a")
    .join(
        dim_broker.select(
            col("SK_BrokerID").cast(LongType()).alias("SK_BrokerID"),
            col("BrokerID").cast(LongType()).alias("broker_id_join")
        ).alias("b"),
        on=(col("a.CA_B_ID") == col("b.broker_id_join")),
        how="left"
    )
    .drop("broker_id_join")
)

broker_null_count = account_with_dims.filter(col("SK_BrokerID").isNull()).count()
print(f"  SK_BrokerID resolved: {account_with_dims.filter(col('SK_BrokerID').isNotNull()).count():,}")
print(f"  SK_BrokerID NULL: {broker_null_count:,}")

print(f"\n  Sample after cascading (Account 4):")
display(
    account_with_dims
    .filter(col("CA_ID") == 4)
    .orderBy("effectivedate")
)

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 5: Assemble Final dim_account DataFrame
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  1. Dedup edge cases (same AccountID + EffectiveDate)
#  2. Recompute version_number after cascading
#  3. Build deterministic SK_AccountID
#  4. Apply final Gold schema
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "="*70)
print("  STEP 5: Assemble Final dim_account")
print("="*70)

# Dedup: enforce one row per (CA_ID, effectivedate)
dedup_window = Window.partitionBy("CA_ID", "effectivedate").orderBy(
    col("enddate").desc(), col("_batch").desc()
)

account_deduped = (
    account_with_dims
    .withColumn("_rn", row_number().over(dedup_window))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

deduped_count = account_deduped.count()
print(f"\n  After dedup (1 per AccountID + EffectiveDate): {deduped_count:,}")

# Recompute version_number + assemble final schema
final_scd_window = Window.partitionBy("CA_ID").orderBy("effectivedate", "_batch")

dim_account_df = (
    account_deduped
    .withColumn("version_number", row_number().over(final_scd_window).cast(LongType()))
    .withColumn("iscurrent", col("enddate") == lit("9999-12-31").cast(DateType()))
    .select(
        ((col("CA_ID") * 10000) + col("version_number")).alias("SK_AccountID"),
        col("CA_ID").alias("AccountID"),
        col("SK_BrokerID").cast(LongType()),
        col("SK_CustomerID").cast(LongType()),
        col("CA_NAME").alias("AccountDesc"),
        col("CA_TAX_ST").cast(ByteType()).alias("TaxStatus"),
        col("CA_ST_ID").alias("Status"),
        col("iscurrent").alias("IsCurrent"),
        col("effectivedate").alias("valid_from"),
        col("enddate").alias("valid_to"),
        col("effectivedate").alias("EffectiveDate"),
        col("enddate").alias("EndDate"),
        col("version_number"),
        col("record_hash"),
        current_timestamp().alias("system_valid_from"),
        lit("9999-12-31T23:59:59").cast(TimestampType()).alias("system_valid_to"),
        col("_batch")
    )
)

final_count = dim_account_df.count()
status = "PASS" if final_count == EXPECTED_ROWS else "MISMATCH"

print(f"\n  +{'─'*54}+")
print(f"  |  Final row count : {final_count:>10,}                   |")
print(f"  |  Expected        : {EXPECTED_ROWS:>10,}                   |")
print(f"  |  Status          : {status:>10}                   |")
print(f"  +{'─'*54}+")

print("\n  Schema:")
for name, dtype in dim_account_df.dtypes:
    print(f"     {name:22s} {dtype}")

print("\n  Sample final dim_account (first 10 rows):")
display(dim_account_df.orderBy("AccountID", "EffectiveDate").limit(10))

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 6: Write to Gold
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "="*70)
print("  STEP 6: Write to Gold (Idempotent)")
print("="*70)

if spark.catalog.tableExists(TARGET_TABLE):
    print(f"\n  Mode: MERGE (table exists)")
    print(f"  Key : (AccountID, EffectiveDate)")
    
    delta_target = DeltaTable.forName(spark, TARGET_TABLE)
    delta_target.alias("tgt").merge(
        dim_account_df.alias("src"),
        "tgt.AccountID = src.AccountID AND tgt.EffectiveDate = src.EffectiveDate"
    ).whenMatchedUpdateAll(
    ).whenNotMatchedInsertAll(
    ).execute()
    
    print("  MERGE complete.")
else:
    print(f"\n  Mode: CREATE (first run)")
    print(f"  Table: {TARGET_TABLE}")
    
    dim_account_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(TARGET_TABLE)
    
    print("  CREATE complete.")

# Verify target count
target_count = spark.table(TARGET_TABLE).count()
status = "PASS" if target_count == EXPECTED_ROWS else "FAIL"
print(f"\n  Target count: {target_count:,} (expected: {EXPECTED_ROWS:,}) [{status}]")

# --- Table Version Info ---
print("\n  Delta Table Version History:")
display(spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 5"))

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  STEP 7: Data Quality Validation
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "="*70)
print("  STEP 7: Data Quality Validation")
print("="*70)

gold_acct = spark.table(TARGET_TABLE)

results = []

# DQ-1: Row Count
row_count = gold_acct.count()
p = row_count == EXPECTED_ROWS
results.append(("DQ-1", "Row Count", f"{row_count:,} vs {EXPECTED_ROWS:,}", "PASS" if p else "FAIL"))

# DQ-2: PK Uniqueness
sk_dups = gold_acct.groupBy("SK_AccountID").count().filter(col("count") > 1).count()
p = sk_dups == 0
results.append(("DQ-2", "SK_AccountID Unique", f"{sk_dups} duplicates", "PASS" if p else "FAIL"))

# DQ-3: SK Not Null
sk_nulls = gold_acct.filter(col("SK_AccountID").isNull()).count()
p = sk_nulls == 0
results.append(("DQ-3", "SK_AccountID Not Null", f"{sk_nulls} nulls", "PASS" if p else "FAIL"))

# DQ-4: SCD-2 IsCurrent uniqueness
current_counts = (
    gold_acct.filter(col("IsCurrent") == True)
    .groupBy("AccountID").count()
    .filter(col("count") > 1).count()
)
p = current_counts == 0
results.append(("DQ-4", "SCD-2 IsCurrent (1 per acct)", f"{current_counts} violations", "PASS" if p else "FAIL"))

# DQ-5: Valid Status Codes
invalid_status = gold_acct.filter(~col("Status").isin("ACTV", "INAC")).count()
p = invalid_status == 0
results.append(("DQ-5", "Valid Status Codes", f"{invalid_status} invalid", "PASS" if p else "FAIL"))

# DQ-6: SK_CustomerID referential integrity
cust_null = gold_acct.filter(col("SK_CustomerID").isNull()).count()
results.append(("DQ-6", "SK_CustomerID NOT NULL", f"{cust_null} nulls", "PASS" if cust_null == 0 else "WARN"))

# DQ-7: SK_BrokerID referential integrity
broker_null = gold_acct.filter(col("SK_BrokerID").isNull()).count()
results.append(("DQ-7", "SK_BrokerID NOT NULL", f"{broker_null} nulls", "PASS" if broker_null == 0 else "WARN"))

# DQ-8: Max version
max_ver = gold_acct.groupBy("AccountID").agg(_max("version_number").alias("mv")).agg(_max("mv")).collect()[0][0]
results.append(("DQ-8", "Max version_number", str(max_ver), "INFO"))

# Print results as a clean table
print("\n  {:<6} {:<32} {:<25} {}".format("Check", "Description", "Result", "Status"))
print("  " + "-"*75)
for check_id, desc, result, status in results:
    print(f"  {check_id:<6} {desc:<32} {result:<25} {status}")

print("\n  Table Detail:")
display(spark.sql(f"DESCRIBE DETAIL {TARGET_TABLE}"))

In [0]:
# ============================================================
# STEP 8: Operations Logging
# ============================================================

print("\n[Step 8] Operations Logging")
print("-"*70)

try:
    # Get the run_id from the source pipeline
    run_id = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.account").select("_run_id").first()[0]
    
    log_pipeline_recon(
        spark=spark,
        run_id=run_id,
        batch_id="ALL",
        domain="ACCOUNT",
        table_name="dim_account",
        source_layer="bronze",
        target_layer="gold",
        source_count=total_actions,
        target_count=target_count
    )
    
    log_audit_event(
        spark=spark,
        run_id=run_id,
        batch="ALL",
        layer="gold",
        table_name="dim_account",
        operation="MERGE" if spark.catalog.tableExists(TARGET_TABLE) else "CREATE",
        rows_affected=target_count
    )
    
    print(f"  \u2713 log_pipeline_recon: source={total_actions:,} \u2192 target={target_count:,}")
    print(f"  \u2713 log_audit_event: operation=MERGE/CREATE, rows={target_count:,}")
except Exception as e:
    print(f"  [WARN] Operations logging failed: {e}")
    print(f"  (Non-blocking - gold table was written successfully)")

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  VERIFICATION: Final State Summary
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "="*70)
print("  VERIFICATION: gold.dim_account Final State")
print("="*70)

final_df = spark.table(TARGET_TABLE)

print(f"\n  Total rows: {final_df.count():,} (expected: {EXPECTED_ROWS:,})")

print("\n  Status Distribution:")
final_df.groupBy("Status").count().orderBy("Status").show()

print("  IsCurrent Distribution:")
final_df.groupBy("IsCurrent").count().show()

print("  Batch Distribution:")
final_df.groupBy("_batch").count().orderBy("_batch").show()

print("  Version Distribution:")
final_df.groupBy("version_number").count().orderBy("version_number").show()

# Show a multi-version account to demonstrate proper SCD-2
print("  Multi-version account example (Account 4):")
display(
    final_df
    .filter(col("AccountID") == 4)
    .orderBy("EffectiveDate")
    .select(
        "SK_AccountID", "AccountID", "SK_BrokerID", "SK_CustomerID",
        "AccountDesc", "TaxStatus", "Status", "IsCurrent",
        "EffectiveDate", "EndDate", "version_number"
    )
)